<a href="https://colab.research.google.com/github/KarthikK04042006/SEL_LAB/blob/main/SEL_LAB_3_4_5_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

def fitness(x):
    return -(x - 5) ** 2 + 25

def create_individual():
    return random.uniform(-10, 10)

def mutate(x, rate=0.1):
    if random.random() < rate:
        x += random.uniform(-1, 1)
    return x

def crossover(a, b):
    t = random.random()
    return t * a + (1 - t) * b

def select(pop):
    k = 3
    c = random.sample(pop, k)
    return max(c, key=fitness)

def genetic_algorithm(pop_size=50, generations=100):
    pop = [create_individual() for _ in range(pop_size)]
    for i in range(generations):
        new_pop = []
        for _ in range(pop_size):
            p1 = select(pop)
            p2 = select(pop)
            child = crossover(p1, p2)
            child = mutate(child)
            new_pop.append(child)
        pop = new_pop
        current_best = max(pop, key=fitness)
        print(f"Generation {i+1}: Best individual = {current_best:.4f}, Fitness = {fitness(current_best):.4f}")
    return max(pop, key=fitness)

best = genetic_algorithm(generations=10)
print(best, fitness(best))

Generation 1: Best individual = 4.9943, Fitness = 25.0000
Generation 2: Best individual = 4.9943, Fitness = 25.0000
Generation 3: Best individual = 4.9931, Fitness = 25.0000
Generation 4: Best individual = 5.0011, Fitness = 25.0000
Generation 5: Best individual = 4.9995, Fitness = 25.0000
Generation 6: Best individual = 5.0000, Fitness = 25.0000
Generation 7: Best individual = 5.0000, Fitness = 25.0000
Generation 8: Best individual = 5.0001, Fitness = 25.0000
Generation 9: Best individual = 5.0001, Fitness = 25.0000
Generation 10: Best individual = 5.0000, Fitness = 25.0000
4.999985113380898 24.99999999977839


In [ ]:
import numpy as np

def gwo(obj_func, lb, ub, dim, pop_size, max_iter):
    alpha_pos = np.zeros(dim)
    alpha_score = float("inf")
    beta_pos = np.zeros(dim)
    beta_score = float("inf")
    delta_pos = np.zeros(dim)
    delta_score = float("inf")

    pos = np.random.uniform(lb, ub, (pop_size, dim))

    for l in range(max_iter):
        for i in range(pop_size):
            pos[i, :] = np.clip(pos[i, :], lb, ub)
            fitness = obj_func(pos[i, :])

            if fitness < alpha_score:
                alpha_score = fitness
                alpha_pos = pos[i, :].copy()
            elif fitness < beta_score:
                beta_score = fitness
                beta_pos = pos[i, :].copy()
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = pos[i, :].copy()

        if l < 10:
            print(f"Iteration {l+1}: Best Score = {alpha_score}")

        a = 2 - l * (2 / max_iter)

        for i in range(pop_size):
            for j in range(dim):
                r1, r2 = np.random.random(), np.random.random()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - pos[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                r1, r2 = np.random.random(), np.random.random()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - pos[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                r1, r2 = np.random.random(), np.random.random()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - pos[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                pos[i, j] = (X1 + X2 + X3) / 3

    return alpha_pos, alpha_score

def sphere_function(x):
    return np.sum(x**2)

best_pos, best_score = gwo(sphere_function, -10, 10, 5, 20, 100)
print("-" * 30)
print(f"Final Best Score: {best_score}")

Iteration 1: Best Score = 111.17175763753158
Iteration 2: Best Score = 29.18954896858421
Iteration 3: Best Score = 25.978894125581448
Iteration 4: Best Score = 15.166439467143942
Iteration 5: Best Score = 10.399447499933176
Iteration 6: Best Score = 2.448673344105528
Iteration 7: Best Score = 0.4974418266014173
Iteration 8: Best Score = 0.4974418266014173
Iteration 9: Best Score = 0.17859587017400042
Iteration 10: Best Score = 0.17859587017400042
------------------------------
Final Best Score: 5.958524631740403e-16


In [ ]:
import numpy as np

def abc(obj_func, lb, ub, dim, n_bees, max_iter, limit):
    pos = np.random.uniform(lb, ub, (n_bees, dim))
    fitness = np.array([obj_func(p) for p in pos])
    trial = np.zeros(n_bees)

    best_idx = np.argmin(fitness)
    best_pos = pos[best_idx].copy()
    best_score = fitness[best_idx]

    for l in range(max_iter):
        for i in range(n_bees):
            phi = np.random.uniform(-1, 1, dim)
            k = np.random.randint(0, n_bees)
            while k == i:
                k = np.random.randint(0, n_bees)

            new_pos = pos[i] + phi * (pos[i] - pos[k])
            new_pos = np.clip(new_pos, lb, ub)
            new_fitness = obj_func(new_pos)

            if new_fitness < fitness[i]:
                pos[i] = new_pos
                fitness[i] = new_fitness
                trial[i] = 0
            else:
                trial[i] += 1

        probs = (1 / (1 + fitness)) / np.sum(1 / (1 + fitness))
        for i in range(n_bees):
            if np.random.random() < probs[i]:
                phi = np.random.uniform(-1, 1, dim)
                k = np.random.randint(0, n_bees)
                while k == i:
                    k = np.random.randint(0, n_bees)

                new_pos = pos[i] + phi * (pos[i] - pos[k])
                new_pos = np.clip(new_pos, lb, ub)
                new_fitness = obj_func(new_pos)

                if new_fitness < fitness[i]:
                    pos[i] = new_pos
                    fitness[i] = new_fitness
                    trial[i] = 0
                else:
                    trial[i] += 1

        for i in range(n_bees):
            if trial[i] > limit:
                pos[i] = np.random.uniform(lb, ub, dim)
                fitness[i] = obj_func(pos[i])
                trial[i] = 0

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_score:
            best_score = fitness[current_best_idx]
            best_pos = pos[current_best_idx].copy()

        if l < 10:
            print(f"Iteration {l+1}: Best Score = {best_score}")

    return best_pos, best_score

def sphere_function(x):
    return np.sum(x**2)

best_p, best_s = abc(sphere_function, -10, 10, 5, 20, 100, 10)
print("-" * 30)
print(f"Final Best Score: {best_s}")

Iteration 1: Best Score = 58.05835605788336
Iteration 2: Best Score = 58.05835605788336
Iteration 3: Best Score = 39.22039085220052
Iteration 4: Best Score = 39.22039085220052
Iteration 5: Best Score = 22.315587829648386
Iteration 6: Best Score = 22.315587829648386
Iteration 7: Best Score = 22.315587829648386
Iteration 8: Best Score = 17.448169083624794
Iteration 9: Best Score = 8.902742188239873
Iteration 10: Best Score = 8.902742188239873
------------------------------
Final Best Score: 0.00021866113074701143


In [ ]:
import numpy as np

def firefly_algorithm(obj_func, lb, ub, dim, n_fireflies, max_iter, alpha=0.5, beta0=1.0, gamma=1.0):
    pos = np.random.uniform(lb, ub, (n_fireflies, dim))
    fitness = np.array([obj_func(p) for p in pos])

    for l in range(max_iter):
        for i in range(n_fireflies):
            for j in range(n_fireflies):
                if fitness[j] < fitness[i]:
                    r = np.linalg.norm(pos[i] - pos[j])
                    beta = beta0 * np.exp(-gamma * r**2)

                    e = np.random.uniform(-0.5, 0.5, dim)
                    pos[i] = pos[i] + beta * (pos[j] - pos[i]) + alpha * e
                    pos[i] = np.clip(pos[i], lb, ub)
                    fitness[i] = obj_func(pos[i])

        best_idx = np.argmin(fitness)
        best_score = fitness[best_idx]

        if l < 10:
            print(f"Iteration {l+1}: Best Score = {best_score}")

    best_idx = np.argmin(fitness)
    return pos[best_idx], fitness[best_idx]

def sphere_function(x):
    return np.sum(x**2)

best_p, best_s = firefly_algorithm(sphere_function, -10, 10, 5, 20, 100)
print("-" * 30)
print(f"Final Best Score: {best_s}")

Iteration 1: Best Score = 56.682674844510046
Iteration 2: Best Score = 56.682674844510046
Iteration 3: Best Score = 56.682674844510046
Iteration 4: Best Score = 56.682674844510046
Iteration 5: Best Score = 56.682674844510046
Iteration 6: Best Score = 56.682674844510046
Iteration 7: Best Score = 56.682674844510046
Iteration 8: Best Score = 56.682674844510046
Iteration 9: Best Score = 56.682674844510046
Iteration 10: Best Score = 56.682674844510046
------------------------------
Final Best Score: 56.682674844510046


In [ ]:
import numpy as np

def pso(obj_func, lb, ub, dim, n_particles, max_iter, w=0.5, c1=1.5, c2=1.5):
    pos = np.random.uniform(lb, ub, (n_particles, dim))
    vel = np.zeros((n_particles, dim))

    pbest_pos = pos.copy()
    pbest_score = np.array([obj_func(p) for p in pos])

    gbest_idx = np.argmin(pbest_score)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_score = pbest_score[gbest_idx]

    for l in range(max_iter):
        for i in range(n_particles):
            r1, r2 = np.random.random(dim), np.random.random(dim)
            vel[i] = (w * vel[i] +
                      c1 * r1 * (pbest_pos[i] - pos[i]) +
                      c2 * r2 * (gbest_pos - pos[i]))

            pos[i] = pos[i] + vel[i]
            pos[i] = np.clip(pos[i], lb, ub)

            fitness = obj_func(pos[i])

            if fitness < pbest_score[i]:
                pbest_score[i] = fitness
                pbest_pos[i] = pos[i].copy()

                if fitness < gbest_score:
                    gbest_score = fitness
                    gbest_pos = pos[i].copy()

        if l < 10:
            print(f"Iteration {l+1}: Best Score = {gbest_score}")

    return gbest_pos, gbest_score

def sphere_function(x):
    return np.sum(x**2)

best_p, best_s = pso(sphere_function, -10, 10, 5, 20, 100)
print("-" * 30)
print(f"Final Best Score: {best_s}")

Iteration 1: Best Score = 24.122755655554982
Iteration 2: Best Score = 17.016945590029355
Iteration 3: Best Score = 5.781522095635098
Iteration 4: Best Score = 1.1325435275229172
Iteration 5: Best Score = 1.1325435275229172
Iteration 6: Best Score = 1.1325435275229172
Iteration 7: Best Score = 1.1325435275229172
Iteration 8: Best Score = 0.20274221876046886
Iteration 9: Best Score = 0.20274221876046886
Iteration 10: Best Score = 0.040663729063747905
------------------------------
Final Best Score: 6.384110323530163e-17
